In [2]:
import pymysql
try:
    print("before connecting")
    connection = pymysql.connect(
        #host="192.168.10.143",
        host="localhost",
        user="root",
        password="Agnext@123",
        #database="mnir_ag001",
        database="vanshita_ias",
        #port=3310,
        port=3306,
        #charset="utf8mb4",
        charset="utf8",
        cursorclass=pymysql.cursors.DictCursor
    )
    print("after connecting", connection)
    print("Connection successful!")
except Exception as e:
    print(" Connection failed:", e) 

before connecting
after connecting <pymysql.connections.Connection object at 0x0000023CCEDC2120>
Connection successful!


In [2]:
# reading ref file
#!pip install pandas
#!pip install openpyxl
import pandas as pd
ref_file=pd.read_excel(r'C:\Users\Agnext\Downloads\Ref.xlsx')
ref_file.head(3)

,Sample ID,Moisture,FFA Result,AV Result
0,RBO-13643,0.67,15.13,30.10
1,RBO-13644,0.83,12.72,25.30
2,RBO-13645,0.72,15.59,31.01


In [3]:
import os
folder_path = r'C:\Users\Agnext\Downloads\converted\converted'
for file in os.listdir(folder_path):
    #print(file)
    full_path= os.path.join(folder_path, file)
    #print(full_path)
    for csv_file in os.listdir(full_path):
        #print(csv_file)
        new_path = os.path.join(full_path, csv_file)
        #print(new_path)

In [4]:

a=r'C:\Users\Agnext\Downloads\converted\converted\RBO-13643\RBO-1364320250127-05046_164512_RBO-1364320250127-05046_0_3-2_20250127111754.csv'
a.split('_')[2][0:-14]

'RBO-13643'

In [5]:

a=r'C:\Users\Agnext\Downloads\converted\converted\RBO-13643\RBO-1364320250127-05046_164512_RBO-1364320250127-05046_0_3-2_20250127111754.csv'
a.split('_')[4].split('-')[1]

'2'

In [6]:
cursor=connection.cursor()


In [7]:
#db helper function
def get_latest_sample_id():
        cursor.execute("SELECT sample_id FROM sample ORDER BY sample_id DESC LIMIT 1;")
        row = cursor.fetchone()
        return str(int(row["sample_id"]) + 1)


In [8]:
result = {}
for file in os.listdir(folder_path):
    full_path = os.path.join(folder_path, file)
    if os.path.isfile(full_path) and file.endswith(".csv"):
        sample_number = (file.split("_")[-2]).split("-")[-1]
        sample_name = file.split("_")[0][:-14]
        if sample_name not in result:
            result[sample_name] = {}
        result[sample_name][sample_number] = [
            pd.read_csv(full_path, skiprows=1, nrows=16),
            pd.read_csv(full_path, skiprows=18),
        ]
    elif os.path.isdir(full_path):
        for f in os.listdir(full_path):
            if f.endswith(".csv"):
                fp = os.path.join(full_path, f)
                sample_number = (f.split("_")[-2]).split("-")[-1]
                sample_name = f.split("_")[0][:-14]
                if sample_name not in result:
                    result[sample_name] = {}
                result[sample_name][sample_number] = [
                    pd.read_csv(fp, skiprows=1, nrows=16),
                    pd.read_csv(fp, skiprows=18),
                ]
print(result)


{'RBO-13643': {'1': [                            Head: Reference  Sample
0         System Temp hundredths:      3268  3301.0
1       Detector Temp hundredths:         0     0.0
2            Humidity hundredths:      4126  4079.0
3                        Lamp PD:       105   104.0
4                     Product SN:  HV6Q625U     NaN
5                  Serial Number:   5B1C061     NaN
6               Scan Config Name:   IAS5100     NaN
7               Scan Config Type:         2     NaN
8         Start wavelength in nm:       900     NaN
9           End wavelength in nm:      1700     NaN
10           Pattern Pixel Width:        11     NaN
11                 Exposure Time:         0     NaN
12            Measurement points:       420     NaN
13  Number of Measurement points:       801     NaN
14                   Num Repeats:        90    90.0
15                      PGA Gain:        64    64.0,      Wavelength  Absorbance  Reference Signal  Sample Signal  \
0           900    0.343155   

In [9]:
all_samples=[]
for sample_name in result:
    sample_id = get_latest_sample_id()
    all_samples.append(sample_id)
    model_wavemin = result[sample_name]["1"][0].loc[
        result[sample_name]["1"][0]["Head:"].str.contains("Start wavelength"),
        "Reference"
    ].values[0]
    model_wavemax = result[sample_name]["1"][0].loc[
        result[sample_name]["1"][0]["Head:"].str.contains("End wavelength"),
        "Reference"
    ].values[0]
 


In [10]:
import pymysql
import pandas as pd
import os
import re

import pymysql
try:
    print("before connecting")
    connection = pymysql.connect(
        host="192.168.10.143",
        user="root",
        password="Agnext@123",
        database="mnir_ag001",
        port=3310,
        charset="utf8mb4",
        cursorclass=pymysql.cursors.DictCursor
    )
    print("after connecting", connection)
    print("Connection successful!")
except Exception as e:
    print(" Connection failed:", e) 
    
# ---------------------------------------------
# DB HELPER FUNCTIONS

def get_latest_sample_id():
    cursor.execute("SELECT sample_id FROM sample ORDER BY sample_id DESC LIMIT 1;")
    row = cursor.fetchone()
    return str(int(row["sample_id"]) + 1)

def get_project_sample_id():
    cursor.execute("SELECT id FROM project_sample ORDER BY id DESC LIMIT 1;")
    row = cursor.fetchone()
    return str(int(row["id"]) + 1)

def get_latest_project_id():
    cursor.execute("SELECT project_id FROM project ORDER BY project_id DESC LIMIT 1;")
    row = cursor.fetchone()
    return str(int(row["project_id"]) + 1)



def get_id_target_name(target):
    cursor.execute("SELECT * FROM content_dictionary;")
    df = pd.DataFrame(cursor.fetchall())
    df["content_name"] = df["content_name"].str.lower()
    target_lower = target.lower()

    # Check if target exists
    if target_lower not in df["content_name"].values:
        # Target not found → insert it into DB
        cursor.execute("INSERT INTO content_dictionary (content_name) VALUES (%s);",(target,))
        connection.commit()

        # Fetch updated table
        cursor.execute("SELECT * FROM content_dictionary;")
        df = pd.DataFrame(cursor.fetchall())
        df["content_name"] = df["content_name"].str.lower()

    # Now fetch the ID safely
    target_id = int(df.loc[df.content_name == target_lower, "id"].values[0])
    return target_id



def extract_sample_id_and_samplename(filename):
    """
    Extracts the sample ID and batch based on pattern occurrence.
    """
    filename=filename.replace(".csv","")

    # Find the pattern
    pattern = r"_0_(5|3|1)-(1|2|3|4|5)"

    match = re.search(pattern, filename)
    if not match:
        raise ValueError(f"Filename does not match expected pattern: {filename}")

    # Identify the batch (5 characters before the pattern)
    start_index = match.start()
    current_batch = filename[start_index - 5:start_index]

    # Check how many times this batch appears
    occurrence_count = filename.count(current_batch)

    # --- Single Occurrence (Device Format) ---
    if occurrence_count == 1:

        # Find position of the batch
        sample_number= filename[-1]
        pos = filename.find(current_batch)
        if current_batch not in filename:
            raise ValueError(f"Batch '{current_batch}' not found in filename: {filename}")

        # Take everything up to the end of the batch
        prefix_end = pos + len(current_batch)
        extracted = filename[:prefix_end]

        # Split by underscore.
        parts = extracted.split('_')

        # Safety check: Ensure split actually created enough parts
        if len(parts) < 2:
            raise ValueError(f"Expected at least 2 parts, got {len(parts)}: {parts}")

        # take second element and remove last 14 chars
        final_sample_id = parts[1][:-14]

        return final_sample_id,sample_number

    # --- Multiple Occurrences (Qualix Format) ---
    else:
        # Find the FIRST occurrence
        sample_number=filename.split('_')[-2].split('-')[-1]
        pos = filename.find(current_batch)
        if current_batch not in filename:
            raise ValueError(f"Batch '{current_batch}' not found in filename: {filename}")

        prefix_end = pos + len(current_batch)
        extracted = filename[:prefix_end]

        if len(extracted) < 14:
            raise ValueError(f"Expected at least 14 extracted values, got {len(extracted)}: {extracted}")

        # remove last 14 characters
        final_sample_id = extracted[:-14]
        # print(final_sample_id)

        return final_sample_id,sample_number




before connecting
after connecting <pymysql.connections.Connection object at 0x00000186AA40D7F0>
Connection successful!


In [11]:
sampleid_folder_path=r'C:\Users\Agnext\Desktop\ias_sw\converted\converted',
reference_file_path=r'C:\Users\Agnext\Desktop\ias_sw\converted\Ref.xlsx',
project_name="Automation IAS Project1"
target="Moisture"

In [12]:
print(reference_file_path)

('C:\\Users\\Agnext\\Desktop\\ias_sw\\converted\\Ref.xlsx',)


In [13]:
# ---------------------------------------------
# READ REFERENCE FILE
# ---------------------------------------------
file_ext = os.path.splitext(reference_file_path)[1].lower()
print(file_ext)

if file_ext in ['.csv']:
    refdf = pd.read_csv(reference_file_path)
elif file_ext in ['.xls', '.xlsx']:
    refdf = pd.read_excel(reference_file_path)
else:
    raise ValueError(f"Unsupported reference file format: {file_ext}")



TypeError: expected str, bytes or os.PathLike object, not tuple

In [ ]:
# ---------------------------------------------
# SCAN SAMPLE FOLDER
# ---------------------------------------------
result = {}

for file in os.listdir(sampleid_folder_path):
    full_path = os.path.join(sampleid_folder_path, file)

    # ---------- CSV directly inside main folder ----------
    if os.path.isfile(full_path) and file.endswith(".csv"):
        try:
            sample_name, sample_number = extract_sample_id_and_samplename(file)
        except ValueError as e:
            raise RuntimeError(f"Sample extraction failed for {file}: {e}")

        if sample_name not in result:
            result[sample_name] = {}

        result[sample_name][sample_number] = [
            pd.read_csv(full_path, skiprows=1, nrows=16),
            pd.read_csv(full_path, skiprows=18),
        ]

    # ---------- CSVs inside subfolders ----------
    elif os.path.isdir(full_path):
        for f in os.listdir(full_path):
            print(f)
            if f.endswith(".csv"):
                fp = os.path.join(full_path, f)

                try:
                    sample_name, sample_number = extract_sample_id_and_samplename(f)
                    # print("************************_______________________")
                    # print("************************_______________________")
                    # print(sample_number)
                    # print("************************_______________________")
                    # print("************************_______________________")
                except ValueError as e:
                    raise RuntimeError(f"Sample extraction failed for {f}: {e}")

                if sample_name not in result:
                    result[sample_name] = {}

                result[sample_name][sample_number] = [
                    pd.read_csv(fp, skiprows=1, nrows=16),
                    pd.read_csv(fp, skiprows=18),
                ]


# ---------------------------------------------
# PROCESS EACH SAMPLE
# ---------------------------------------------
all_samples=[]
for sample_name in result:
    sample_id = get_latest_sample_id()
    print(sample_id, "sample ID")
    all_samples.append(sample_id)

    model_wavemin = result[sample_name]["1"][0].loc[
        result[sample_name]["1"][0]["Head:"].str.contains("Start wavelength"),
        "Reference"
    ].values[0]

    model_wavemax = result[sample_name]["1"][0].loc[
        result[sample_name]["1"][0]["Head:"].str.contains("End wavelength"),
        "Reference"
    ].values[0]

    property_name = get_id_target_name(target)
    property_value = refdf.loc[refdf["Sample ID"] == sample_name, target].values[0]

    # ---------------------------------------------
    # INSERT INTO sample TABLE
    # ---------------------------------------------
    insert_sample = """
    INSERT INTO sample (
        sample_id, sample_name, model_num, model_wavemin, model_wavemax,
        model_wavepath, model_method, sample_status,
        property_name1, property_value1,
        create_person, create_time, sample_state
    )
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,NOW(),%s)
    """

    cursor.execute(insert_sample, (
        sample_id, sample_name, 0, str(model_wavemin), str(model_wavemax),
        '1', '0', '0',
        str(property_name), str(property_value),
        'script', '1'
    ))

    # ---------------------------------------------
    # INSERT INTO model_data TABLE
    # ---------------------------------------------
    insert_model = """
    INSERT INTO model_data (
        sample_id, model_sno, model_order, device_id,
        model_length, wave, absorb, system_temp,create_time
    ) VALUES (%s,%s,%s,%s,%s,%s,%s,%s, NOW())
    """

    for sample_number in result[sample_name]:

        df_meas = result[sample_name][sample_number][1]
        wave = ",".join(df_meas["Wavelength"].astype(str))
        absorb = ",".join(df_meas["Absorbance"].astype(str))

        model_order = sample_number

        cursor.execute(insert_model, (
            sample_id, "", model_order, "30001",
            "801", wave, absorb, "0"
        ))

    print(f"Inserted sample: {sample_name}")

# ---------------------------------------------
# INSERT PROJECT ENTRY
# ---------------------------------------------
project_id = get_latest_project_id()

insert_project = """
INSERT INTO project (
    project_id, project_name, sample_type, analysis_type,
    analysis_object, project_progress, project_remark,
    create_person, create_time, modify_time, project_state
)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,NOW(),NULL,%s)
"""

cursor.execute(insert_project, (
    project_id, project_name, "0", "1",
    str(property_name), "0", "",
    "by_automation_script", "1"
))

for sid in all_samples:
    insert_project_sample = """
    INSERT INTO project_sample (id,project_id,sample_id,new_id,new_name)
    VALUES (%s, %s, %s, %s, %s)"""
    id_var=get_project_sample_id()

    cursor.execute(insert_project_sample,(id_var, project_id, sid, "", ""))



# Commit all DB changes
connection.commit()
print("\nALL DONE — Data inserted successfully!\n")




.~lock.RBO-1364320250127-05046_164512_RBO-1364320250127-05046_0_3-1_20250127111753.csv#
RBO-1364320250127-05046_164512_RBO-1364320250127-05046_0_3-1_20250127111753.csv
RBO-1364320250127-05046_164512_RBO-1364320250127-05046_0_3-2_20250127111754.csv
RBO-1364320250127-05046_164512_RBO-1364320250127-05046_0_3-3_20250127111754.csv
RBO-1364420250127-05047_171030_RBO-1364420250127-05047_0_3-1_20250127114310.csv
RBO-1364420250127-05047_171030_RBO-1364420250127-05047_0_3-2_20250127114310.csv
RBO-1364420250127-05047_171030_RBO-1364420250127-05047_0_3-3_20250127114310.csv
RBO-1364520250127-05048_171831_RBO-1364520250127-05048_0_3-1_20250127115113.csv
RBO-1364520250127-05048_171831_RBO-1364520250127-05048_0_3-2_20250127115113.csv
RBO-1364520250127-05048_171831_RBO-1364520250127-05048_0_3-3_20250127115113.csv
RBO-1364620250127-05049_172950_RBO-1364620250127-05049_0_3-1_20250127120232.csv
RBO-1364620250127-05049_172950_RBO-1364620250127-05049_0_3-2_20250127120233.csv
RBO-1364620250127-05049_172950_R

IntegrityError: (1062, "Duplicate entry '100000' for key 'PRIMARY'")

In [ ]:
cursor=connection.cursor()